In [ ]:
!pip install "transformers"
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
#random sampling
#train_data = train_data.sample(n=4000,random_state=42).reset_index(drop=True)
#val_data = val_data.sample(n=500,random_state = 42).reset_index(drop=True)

# data pre-processing


In [ ]:
import re

def clean_data(text):

  text=re.sub(r"\r\n"," ", text) #lines
  text=re.sub(r"\s+"," ",text) #remove spaces
  text=re.sub(r"<.*?>"," ",text) #html tags

  emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )

  text = emoji_pattern.sub(r'', text)

  text = text.strip().lower()

  return text

In [ ]:
train_data ["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data.sample(5)

,id,dialogue,summary
11918,13715929,wilkes: anybody been to jordan? taylor: you me...,wilkes wants to go to jordan as his girlfriend...
218,13829213,monica: i've got it! josh: tell me! monica: i ...,josh wants to propose to his girlfriend next m...
13661,13862880,"david: hi, how are you? janette: all good, hbu...",david bought his tickets. janette buys her tic...
1852,13828464,niidia: waiting for the quiz results has been ...,niidia is impatiently waiting for the quiz res...
5485,13729984,karen: did you hear nick and kasia went out to...,"nick and kasia went out to brunch yesterday, b..."


# Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#raw data => tokens for fine tuning and model training

def tokenize(data):
  inputs = tokenizer(data["dialogue"],padding = "max_length",max_length=512, truncation=True)
  targets = tokenizer(data["summary"],padding = "max_length",max_length=150, truncation=True)

  inputs["labels"] = targets["input_ids"] ## token ids =>add to input as labels

  return inputs

In [ ]:
train_dataset = train_data.apply(tokenize,axis =1).tolist()
val_dataset = val_data.apply(tokenize,axis =1).tolist()

In [ ]:
train_dataset[0]

#input ids - dialogue =>token ids
#1 =>EOS(end of sequence)  se after the number for input_ids and labels there is a 1

#attention mask => says in the tokens which are valid values and which are invalid or padding values

#labels ->target =>summary token

{'input_ids': [183, 232, 9, 10, 3, 23, 13635, 5081, 5, 103, 25, 241, 128, 58, 3, 12488, 651, 10, 417, 55, 183, 232, 9, 10, 3, 23, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# working with the Model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

model.to(device)

cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
#training arguements

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch",
    save_strategy= "epoch",

    warmup_steps=500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
#train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.392525,0.341092
2,0.359221,0.330792
3,0.354436,0.324669
4,0.341544,0.323557
5,0.338552,0.321286
6,0.334092,0.320867


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11052, training_loss=0.5016651812818143, metrics={'train_runtime': 4388.5215, 'train_samples_per_second': 20.142, 'train_steps_per_second': 2.518, 'total_flos': 1.1963132515713024e+16, 'train_loss': 0.5016651812818143, 'epoch': 6.0})

In [ ]:
from google.colab import files
import shutil

# Create zip
shutil.make_archive("results", 'zip', "/content/results")

# Download zip
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "saved_summary_model",
    'zip',
    "/content/saved_summary_model"
)

files.download("saved_summary_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# test the core logic for summarization


In [ ]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"  #pt=> pytorch tensors
  ).to(device)



  model.to(device)

  #generate the summary => tokens ids

  targets=model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams = 4, #beam search=> compare 4 different summary sequence and return the best
      early_stopping=True
  )

  #convert the token ids to text => decoding
  summary=tokenizer.decode(targets[0],skip_special_tokens=True) # skip-> EOS,SEP

  return summary

In [ ]:
test_dialogue = '''
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)
'''


summary=summarize_dialogue(test_dialogue)

print(summary)

eric likes the train part. he will watch some of his stand-ups on youtube.
